In [11]:
#!pip install tensorflow
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow **IS** using the GPU")
else:
    print("TensorFlow **IS NOT** using the GPU")

Num GPUs Available:  1
TensorFlow **IS** using the GPU


In [12]:
X = tf.constant([
  [0, 0],
  [0, 1],
  [1, 0],
  [1, 1]
], dtype=tf.float32)
Y = tf.constant([[0], [1], [1], [0]], dtype=tf.float32) #tf.constant([0, 1, 1, 0], dtype=tf.float32)

In [13]:
def init_weights(input_size, neuron_num, init_rand):
  #w = tf.Variable(tf.zeros(input_size, neuron_num), name='weights')
  w = tf.Variable(tf.random.uniform((input_size, neuron_num), -init_rand, init_rand, name='weights'))
  #w = tf.Variable(tf.random.normal((input_size, neuron_num), name='weights'))
  b = tf.Variable(tf.random.uniform([neuron_num], -init_rand, init_rand, name='bias'), trainable=True)
  #b = tf.Variable(tf.random.normal([neuron_num], name='bias'))
  return w, b

In [14]:
class Dense(tf.Module):
  def __init__(self, neuron_num, activate='sigmoid'): #сигмоида по умолчанию
    super().__init__()
    self.neuron_num = neuron_num
    self.activate = activate
    self.fl_init = False
  def __call__(self, x): # функтор
    if not self.fl_init:
      self.w, self.b = init_weights(x.shape[-1], self.neuron_num, 1/x.shape[-1])
      self.fl_init = True
    net = tf.matmul(x, self.w) + self.b
    match self.activate:
      case 'sigmoid':
        return tf.math.sigmoid(net)
      case 'linear':
        return net
      case 'relu': #взрыв градиента
        return tf.nn.relu(net)

In [15]:
class SequentialModule(tf.Module):
    def __init__(self, activate='sigmoid'):
        super().__init__()
        self.hidden_layer = Dense(2, activate)
        self.output_layer = Dense(1)
    def __call__(self, x):
        return self.output_layer(self.hidden_layer(x))

In [16]:
@tf.function
def loss(pred, Y):
    return tf.reduce_mean(-(Y * tf.math.log(pred) + (1 - Y) * tf.math.log(1 - pred)))
    #return tf.nn.sigmoid_cross_entropy_with_logits(labels=Y, logits=net)

In [17]:
#@tf.function
def LetsTry(X, Y, activate, l):
  opt = tf.optimizers.Adam(learning_rate=l)
  old_mean = -np.inf
  m = SequentialModule(activate) #модель
  mean_loss = np.inf
  epoch = 0
  epsilon = 5e-3
  while (mean_loss > epsilon):
    old_mean = mean_loss
    mean_loss = 0
    with tf.GradientTape() as t:
      current_loss = loss(m(X), Y)
    gradients = t.gradient(current_loss, m.trainable_variables)
    #print(current_loss)
    opt.apply_gradients(zip(gradients, m.trainable_variables))
    mean_loss = current_loss
    if epoch % 50 == 0:
      print(f"\t\t{activate}\nEpoch: {epoch}, mean_loss: {mean_loss}")
    if abs(mean_loss - old_mean) < 1e-25: #модель не улучшается
      print(f"Модель не сошлась на эпохе {epoch}, активация: {activate}, mean_loss: {mean_loss}")
      break
    if mean_loss < epsilon:
      print(f"Модель СОШЛАСЬ на эпохе {epoch}, активация: {activate}, mean_loss: {mean_loss}")
    epoch +=1
  return m

Линейная функция активации не решает задачу XOR, т.к. выборка линейно не разделима, а лин.акт. = просто x*w - по факту отсутствие активации. По итогу работает только 1 нейрон - выходной (куда по умолчанию установлена сигмоида в качестве активации)

In [18]:
lin = LetsTry(X, Y, 'linear', 1e-2)

		linear
Epoch: 0, mean_loss: 0.6954245567321777
		linear
Epoch: 50, mean_loss: 0.6931533217430115
Модель не сошлась на эпохе 83, активация: linear, mean_loss: 0.6931473612785339


ReLU иногда сходится, иногда нет. Он нестабилен, если числа отрицательны: max(0, x) - они занулятся. Также есть проблема взрыва градиентов при положительных весах.

In [19]:
relu = LetsTry(X, Y, 'relu', 1e-2)

		relu
Epoch: 0, mean_loss: 0.6995059251785278
		relu
Epoch: 50, mean_loss: 0.6527524590492249
		relu
Epoch: 100, mean_loss: 0.5692617893218994
		relu
Epoch: 150, mean_loss: 0.5149809122085571
		relu
Epoch: 200, mean_loss: 0.4942897856235504
		relu
Epoch: 250, mean_loss: 0.4867418110370636
		relu
Epoch: 300, mean_loss: 0.4837033152580261
		relu
Epoch: 350, mean_loss: 0.4815393090248108
		relu
Epoch: 400, mean_loss: 0.4806501567363739
		relu
Epoch: 450, mean_loss: 0.4797790050506592
		relu
Epoch: 500, mean_loss: 0.4793155789375305
		relu
Epoch: 550, mean_loss: 0.47922420501708984
		relu
Epoch: 600, mean_loss: 0.47869613766670227
		relu
Epoch: 650, mean_loss: 0.47875720262527466
		relu
Epoch: 700, mean_loss: 0.4784001111984253
		relu
Epoch: 750, mean_loss: 0.47825103998184204
		relu
Epoch: 800, mean_loss: 0.47834452986717224
		relu
Epoch: 850, mean_loss: 0.47813865542411804
		relu
Epoch: 900, mean_loss: 0.47797152400016785
		relu
Epoch: 950, mean_loss: 0.47791630029678345
Модель не сошла

Сигмоида стабильна всегда, т.к. гладкая, градиенты стабильны, и выход строго от 0 до 1

In [20]:
sig = LetsTry(X, Y, 'sigmoid', 1e-2)

		sigmoid
Epoch: 0, mean_loss: 0.7097207307815552
		sigmoid
Epoch: 50, mean_loss: 0.6930160522460938
		sigmoid
Epoch: 100, mean_loss: 0.6924335360527039
		sigmoid
Epoch: 150, mean_loss: 0.686288595199585
		sigmoid
Epoch: 200, mean_loss: 0.6450169086456299
		sigmoid
Epoch: 250, mean_loss: 0.5838093757629395
		sigmoid
Epoch: 300, mean_loss: 0.47671815752983093
		sigmoid
Epoch: 350, mean_loss: 0.31423333287239075
		sigmoid
Epoch: 400, mean_loss: 0.2120686024427414
		sigmoid
Epoch: 450, mean_loss: 0.15453821420669556
		sigmoid
Epoch: 500, mean_loss: 0.11933853477239609
		sigmoid
Epoch: 550, mean_loss: 0.09595976769924164
		sigmoid
Epoch: 600, mean_loss: 0.0794418528676033
		sigmoid
Epoch: 650, mean_loss: 0.06722049415111542
		sigmoid
Epoch: 700, mean_loss: 0.057853203266859055
		sigmoid
Epoch: 750, mean_loss: 0.050471555441617966
		sigmoid
Epoch: 800, mean_loss: 0.044523097574710846
		sigmoid
Epoch: 850, mean_loss: 0.03964066132903099
		sigmoid
Epoch: 900, mean_loss: 0.03557094931602478
		

In [21]:
for i in range(4):
  print(f'{lin(X).numpy()[i]}\t\t{relu(X).numpy()[i]}\t\t{sig(X).numpy()[i]}')

[0.50033957]		[0.33407098]		[0.00471845]
[0.5002182]		[0.99784935]		[0.9937203]
[0.5004315]		[0.33407098]		[0.9957135]
[0.5003102]		[0.33407098]		[0.00464152]


In [22]:
print('\t\t=', lin.variables)
print('\t\t=', relu.variables)
print('\t\t=', sig.variables)

		= (<tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([-0.42280334,  0.53251445], dtype=float32)>, <tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[ 0.10387719,  0.12195564],
       [-0.28137937, -0.32906222]], dtype=float32)>, <tf.Variable 'Variable:0' shape=(1,) dtype=float32, numpy=array([-0.40569076], dtype=float32)>, <tf.Variable 'Variable:0' shape=(2, 1) dtype=float32, numpy=
array([[-0.46263367],
       [ 0.39707065]], dtype=float32)>)
		= (<tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([-0.14019   , -0.04149581], dtype=float32)>, <tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[-0.4779564 , -1.9903383 ],
       [-0.05629289,  1.9936893 ]], dtype=float32)>, <tf.Variable 'Variable:0' shape=(1,) dtype=float32, numpy=array([-0.68982965], dtype=float32)>, <tf.Variable 'Variable:0' shape=(2, 1) dtype=float32, numpy=
array([[0.32290316],
       [3.4984536 ]], dtype=float32)>)
		= (<tf.Variable 'Variable:0' shape=(

In [23]:
save_path = './saved1'
tf.saved_model.save(sig, save_path)

In [24]:
tf.executing_eagerly()

True